# Embeddings (LO1)

In [5]:
!pip install neo4j


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Given a route, we want to get the top 10 more similar routes to it (grade, lenght, location...)

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from neo4j import GraphDatabase

# Setup the connection to the Neo4j database
uri = "bolt://localhost:7687"
driver = GraphDatabase.driver(uri, auth=("neo4j", "---"))

def fetch_hybrid_data():
    with driver.session() as session:
        # Retrieve route metadata
        query = """
        MATCH (r:Route)
        OPTIONAL MATCH (r)-[:IN_AREA]->(loc:Location)
        RETURN 
            id(r) AS internal_id, 
            r.name AS name, 
            r.rating AS rating_num, 
            r.rating_label AS rating_label, 
            r.length AS length,
            r.lat AS lat,
            r.lon AS lon,
            loc.name AS sector,
            r.embedding_geo_rico AS fastrp_emb
        """
        res = session.run(query)
        return pd.DataFrame([dict(record) for record in res])

# Load data and close connection
df = fetch_hybrid_data()
driver.close()

# Clean and prepare the dataset
# We remove routes without structural embeddings and fill missing values for length and sector
df = df.dropna(subset=['fastrp_emb']).reset_index(drop=True)
df['length'] = df['length'].fillna(30.0)
df['sector'] = df['sector'].fillna("Unknown")

# Apply Min-Max normalization to features to keep them within a 0-1 range
min_lat, max_lat = df['lat'].min(), df['lat'].max()
min_lon, max_lon = df['lon'].min(), df['lon'].max()
max_rating = 12.0
max_length = df['length'].max()

# Build the hybrid embedding vectors
# We combine the 64-dimensional FastRP vector with specific route features
all_vectors = []

for i, row in df.iterrows():
    # Base structural embedding from Neo4j
    v_base = np.array(row['fastrp_emb']) 
    
    # Normalize our numerical features
    r_norm = (row['rating_num'] / max_rating)
    l_norm = (row['length'] / max_length)
    lat_norm = (row['lat'] - min_lat) / (max_lat - min_lat) if max_lat != min_lat else 0.0
    lon_norm = (row['lon'] - min_lon) / (max_lon - min_lon) if max_lon != min_lon else 0.0
    
    # Create the additional feature vector
    # We multiply the rating by 2 to give difficulty higher priority in the similarity calculation
    v_extra = np.array([r_norm * 2.0, l_norm, lat_norm, lon_norm])
    
    # Concatenate the structural and numerical data into a single vector
    v_hybrid = np.concatenate([v_base, v_extra])
    all_vectors.append(v_hybrid)

# Convert the list of vectors into a tensor
x_tensor = torch.tensor(np.array(all_vectors), dtype=torch.float)

# Function to search for and recommend similar climbing routes
def recommend_hybrid(target_id, df, x_tensor, top_k=10):
    try:
        # Locate the index of the route based on its internal Neo4j ID
        idx = df[df['internal_id'] == target_id].index[0]
    except IndexError:
        print("Error: Target ID not found in the dataset.")
        return

    # Extract target vector and calculate cosine similarity against all other routes
    target_v = x_tensor[idx].unsqueeze(0)
    similarities = F.cosine_similarity(target_v, x_tensor)
    
    # Retrieve the top K + 1 results (the first one will be the route itself)
    scores, indices = torch.topk(similarities, k=top_k + 1)
    
    selected = df.iloc[idx]
    print(f"\nTARGET ROUTE: {selected['name']} | Grade: {selected['rating_label']} | Sector: {selected['sector']} | Length: {selected['length']:.1f}m | Coords: ({selected['lat']:.5f}, {selected['lon']:.5f})")
    print("-" * 120)
    print(f"{'SIMILARITY':<10} | {'NAME':<25} | {'GRADE':<8} | {'SECTOR':<18} | {'LENGTH':<7} | {'LAT':<9} | {'LON':<9}")
    print("-" * 120)
    
    # Exclude the target route from its own recommendations
    for i in range(1, len(indices)):
        route_idx = indices[i].item()
        row = df.iloc[route_idx]
        print(f"{scores[i].item():<10.4f} | {row['name'][:25]:<25} | {row['rating_label']:<8} | {row['sector'][:18]:<18} | {row['length']:<7.1f}m | {row['lat']:<9.5f} | {row['lon']:<9.5f}")

# Example execution using a specific internal ID 
target_route_id = 1031
recommend_hybrid(target_route_id, df, x_tensor)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=5, column=13, offset=107>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 107, 'line': 5, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (r:Route)\n        OPTIONAL MATCH (r)-[:IN_AREA]->(loc:Location)\n        RETURN \n            id(r) AS internal_id, \n            r.name AS name, \n            r.rating AS rating_num, \n            r.rating_label AS rating_label, \n            r.length AS length,\n            r.lat AS lat,\n   


TARGET ROUTE: Crescent Arch | Grade: 5.10a | Sector: West Face | Length: 293.7m | Coords: (37.87950, -119.41345)
------------------------------------------------------------------------------------------------------------------------
SIMILARITY | NAME                      | GRADE    | SECTOR             | LENGTH  | LAT       | LON      
------------------------------------------------------------------------------------------------------------------------
0.9999     | Cooke Book                | 5.10a    | West Face          | 500.0  m | 37.87950  | -119.41345
0.9999     | Life in the Cretaceous    | 5.10a    | West Face          | 50.0   m | 37.88460  | -119.40825
0.9986     | Digging in the Dirt       | 5.10a    | West Face          | 140.0  m | 36.54275  | -118.76663
0.9937     | Lock, Stock, and Barrel   | 5.10a    | Murphy Creek       | 50.0   m | 37.85203  | -119.46983
0.9934     | Hyperspace                | 5.10b    | West Face          | 165.0  m | 38.77639  | -120.30850
0.99

# GNN (LO3)

## Regression

We are going to train a GNN (SAGE Regressor) to predict the grade (difficulty) of a route given its characteristics and neighbors.

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from neo4j import GraphDatabase
import pandas as pd
import numpy as np
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

# Initialize Neo4j driver
uri = "bolt://localhost:7687"
driver = GraphDatabase.driver(uri, auth=("neo4j", "---")) 

def fetch_complete_data():
    with driver.session() as session:
        # Fetch structural embeddings and raw route features
        query = """
        MATCH (n)
        OPTIONAL MATCH (n)-[:IN_AREA]->(loc:Location)
        RETURN id(n) AS internal_id, 
               labels(n)[0] AS type,
               n.rating AS y_label,
               n.length AS length,
               n.stars AS stars,
               n.lat AS lat,
               n.lon AS lon,
               n.embedding_gnn_clean AS emb_base
        """
        nodes_res = session.run(query)
        df_nodes = pd.DataFrame([dict(r) for r in nodes_res])

        # Retrieve graph relationships
        edge_query = "MATCH (s)-[]->(t) RETURN id(s) AS source, id(t) AS target"
        edges_res = session.run(edge_query)
        df_edges = pd.DataFrame([dict(r) for r in edges_res])
        
    return df_nodes, df_edges

df_nodes, df_edges = fetch_complete_data()

# Combine physical attributes with 64D structural embeddings
X_list = []
for i, row in df_nodes.iterrows():
    # Handle missing physical data
    phys = [
        row['length'] if pd.notnull(row['length']) else 65.0,
        row['stars'] if pd.notnull(row['stars']) else 2.5,
        row['lat'] if pd.notnull(row['lat']) else 0.0,
        row['lon'] if pd.notnull(row['lon']) else 0.0
    ]
    # Incorporate Neo4j structural vector
    struct = row['emb_base'] if row['emb_base'] is not None else [0.0]*64
    X_list.append(phys + struct)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(np.array(X_list))
x_tensor = torch.tensor(X_scaled, dtype=torch.float)

# Map database IDs to tensor indices
mapping = {node_id: i for i, node_id in enumerate(df_nodes['internal_id'])}

# Extract source-target pairs
edges_list = []
for s, t in zip(df_edges['source'], df_edges['target']):
    if s in mapping and t in mapping:
        edges_list.append([mapping[s], mapping[t]])

# Convert to PyG required format: [2, NumEdges]
edge_index = torch.tensor(edges_list, dtype=torch.long).t().contiguous()

# Define labels and training masks for Route nodes
y = torch.full((len(df_nodes),), -1, dtype=torch.long)
train_mask = torch.zeros(len(df_nodes), dtype=torch.bool)

for i, row in df_nodes.iterrows():
    if row['type'] == 'Route' and pd.notnull(row['y_label']):
        y[i] = int(row['y_label'])
        train_mask[i] = True

data = Data(x=x_tensor, edge_index=edge_index, y=y, train_mask=train_mask)

# GraphSAGE regressor architecture
class ClimbingSAGERegressor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels // 2)
        # Linear layer for single continuous difficulty prediction
        self.out = nn.Linear(hidden_channels // 2, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index).relu()
        return self.out(x)

# Initialize model with 68 input features
model = ClimbingSAGERegressor(in_channels=68, hidden_channels=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

# Training loop using MSE Loss for regression
print("Training GNN Regressor...")
model.train()
for epoch in range(1, 701):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index).squeeze()
    
    # Calculate loss only for nodes with existing labels
    target = data.y[data.train_mask].float()
    loss = F.mse_loss(out[data.train_mask], target)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 70 == 0:
        # Monitor Mean Absolute Error (MAE)
        with torch.no_grad():
            mae = torch.abs(out[data.train_mask] - target).mean()
            print(f'Epoch {epoch:03d} | MSE Loss: {loss.item():.4f} | MAE: {mae.item():.2f} grades')

# Final evaluation
model.eval()
with torch.no_grad():
    final_out = model(data.x, data.edge_index).squeeze()
    final_mae = torch.abs(final_out[data.train_mask] - data.y[data.train_mask].float()).mean()
    
    print(f"\nFINAL REGRESSION RESULTS")
    print(f"Mean Absolute Error (MAE): {final_mae.item():.4f} grades")

# Close driver connection
driver.close()

c:\Users\Propietario\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=16, offset=88>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 88, 'line': 4, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (n)\n        O

Training GNN Regressor...
Epoch 070 | MSE Loss: 8.9271 | MAE: 2.41 grades
Epoch 140 | MSE Loss: 7.9959 | MAE: 2.29 grades
Epoch 210 | MSE Loss: 7.7306 | MAE: 2.26 grades
Epoch 280 | MSE Loss: 7.5998 | MAE: 2.23 grades
Epoch 350 | MSE Loss: 7.4774 | MAE: 2.21 grades
Epoch 420 | MSE Loss: 7.4521 | MAE: 2.21 grades
Epoch 490 | MSE Loss: 7.3599 | MAE: 2.20 grades
Epoch 560 | MSE Loss: 7.2983 | MAE: 2.18 grades
Epoch 630 | MSE Loss: 7.2621 | MAE: 2.18 grades
Epoch 700 | MSE Loss: 7.1652 | MAE: 2.16 grades

FINAL REGRESSION RESULTS
Mean Absolute Error (MAE): 2.1313 grades


## With 3 difficulty classes

Now we'll divide the difficulty grades into 3 different classes (Beginner, Intermediate, Advanced) to make it a classification problem.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from neo4j import GraphDatabase
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

# Initialize Neo4j driver
uri = "bolt://localhost:7687"
driver = GraphDatabase.driver(uri, auth=("neo4j", "---")) 

def fetch_complete_data():
    with driver.session() as session:
        # Retrieve structural embeddings and raw physical features from the graph
        query = """
        MATCH (n)
        OPTIONAL MATCH (n)-[:IN_AREA]->(loc:Location)
        RETURN id(n) AS internal_id, 
               labels(n)[0] AS type,
               n.rating AS rating_num,
               n.length AS length,
               n.stars AS stars,
               n.lat AS lat,
               n.lon AS lon,
               n.embedding_gnn_clean AS emb_base
        """
        nodes_res = session.run(query)
        df_nodes = pd.DataFrame([dict(r) for r in nodes_res])

        # Get all relationships to define the graph edges
        edge_query = "MATCH (s)-[]->(t) RETURN id(s) AS source, id(t) AS target"
        edges_res = session.run(edge_query)
        df_edges = pd.DataFrame([dict(r) for r in edges_res])
        
    return df_nodes, df_edges

df_nodes, df_edges = fetch_complete_data()

# Map granular ratings into 3 main athletic categories
def map_to_category(rating_num):
    if rating_num <= 4: return 0  # Beginner (Up to 5.9)
    if rating_num <= 8: return 1  # Intermediate (5.10a to 5.11a)
    return 2                      # Advanced (5.11b and up)

df_nodes['category'] = df_nodes['rating_num'].apply(lambda x: map_to_category(x) if pd.notnull(x) else -1)

# Build feature matrix X by merging physical data with graph embeddings
X_list = []
for i, row in df_nodes.iterrows():
    # Handle missing values for physical attributes
    phys = [
        row['length'] if pd.notnull(row['length']) else 65.0,
        row['stars'] if pd.notnull(row['stars']) else 2.5,
        row['lat'] if pd.notnull(row['lat']) else 0.0,
        row['lon'] if pd.notnull(row['lon']) else 0.0
    ]
    struct = row['emb_base'] if row['emb_base'] is not None else [0.0]*64
    X_list.append(phys + struct)

# Normalize features to improve GNN convergence
scaler = StandardScaler()
X_scaled = scaler.fit_transform(np.array(X_list))
x_tensor = torch.tensor(X_scaled, dtype=torch.float)

# Map Neo4j internal IDs to tensor indices and create edge_index
mapping = {node_id: i for i, node_id in enumerate(df_nodes['internal_id'])}
edges_list = [[mapping[s], mapping[t]] for s, t in zip(df_edges['source'], df_edges['target']) 
              if s in mapping and t in mapping]
edge_index = torch.tensor(edges_list, dtype=torch.long).t().contiguous()

# Define targets and training masks for Route nodes
y = torch.tensor(df_nodes['category'].values, dtype=torch.long)
train_mask = torch.tensor((df_nodes['type'] == 'Route') & (df_nodes['category'] != -1), dtype=torch.bool)

data = Data(x=x_tensor, edge_index=edge_index, y=y, train_mask=train_mask)

# Deep GraphSAGE model with 3 convolutional layers
class ClimbingSAGE_Deep(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels_1, hidden_channels_2, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels_1)
        self.conv2 = SAGEConv(hidden_channels_1, hidden_channels_2)
        self.conv3 = SAGEConv(hidden_channels_2, out_channels)
        
        # Batch normalization for training stability
        self.bn1 = nn.BatchNorm1d(hidden_channels_1)
        self.bn2 = nn.BatchNorm1d(hidden_channels_2)

    def forward(self, x, edge_index):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=0.4, training=self.training)
        
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=0.4, training=self.training)
        
        return self.conv3(x, edge_index)

model = ClimbingSAGE_Deep(
    in_channels=68, 
    hidden_channels_1=128, 
    hidden_channels_2=64, 
    out_channels=3
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Training loop
print("Starting training: 3-Class Level Classifier...")
model.train()
for epoch in range(1, 1001):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        preds = out[data.train_mask].argmax(dim=1).detach().numpy()
        current_f1 = f1_score(data.y[data.train_mask].numpy(), preds, average='macro')
        print(f'Epoch {epoch:03d} | Loss: {loss.item():.4f} | F1 Score: {current_f1:.4f}')

# Final evaluation and detailed performance metrics
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    preds = logits[data.train_mask].argmax(dim=1).numpy()
    target = data.y[data.train_mask].numpy()
    
    class_names = ["Beginner", "Intermediate", "Advanced"]
    print("\nFINAL PERFORMANCE REPORT (3 CATEGORIES)")
    print(classification_report(target, preds, target_names=class_names))
    print(f"Final F1-Macro (Dice Score): {f1_score(target, preds, average='macro'):.4f}")

driver.close()

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=16, offset=88>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 88, 'line': 4, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (n)\n        OPTIONAL MATCH (n)-[:IN_AREA]->(loc:Location)\n        RETURN id(n) AS internal_id, \n               labels(n)[0] AS type,\n               n.rating AS rating_num,\n               n.length AS length,\n               n.stars AS stars,\n               n.lat AS lat,\n               n.lon 

Starting training: 3-Class Level Classifier...
Epoch 100 | Loss: 0.8865 | F1 Score: 0.5080
Epoch 200 | Loss: 0.8709 | F1 Score: 0.5270
Epoch 300 | Loss: 0.8555 | F1 Score: 0.5415
Epoch 400 | Loss: 0.8488 | F1 Score: 0.5482
Epoch 500 | Loss: 0.8347 | F1 Score: 0.5640
Epoch 600 | Loss: 0.8292 | F1 Score: 0.5661
Epoch 700 | Loss: 0.8135 | F1 Score: 0.5768
Epoch 800 | Loss: 0.8071 | F1 Score: 0.5898
Epoch 900 | Loss: 0.7925 | F1 Score: 0.5846
Epoch 1000 | Loss: 0.7901 | F1 Score: 0.5869

FINAL PERFORMANCE REPORT (3 CATEGORIES)
              precision    recall  f1-score   support

    Beginner       0.74      0.80      0.77      3075
Intermediate       0.62      0.72      0.67      2817
    Advanced       0.68      0.33      0.45      1410

    accuracy                           0.68      7302
   macro avg       0.68      0.62      0.63      7302
weighted avg       0.68      0.68      0.67      7302

Final F1-Macro (Dice Score): 0.6264
